# 02 · Clean, validate and engineer features

Preserve raw observations, validate join cardinality, distinguish missing statistics from zeros, and calculate fighter history using strictly earlier dates. See DATA_NOTES.md for every important decision.

In [1]:
from pathlib import Path
import sys
import io
import pandas as pd
import numpy as np
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_loader import load_raw, profile_sources, verify_raw, TABLES
from src.pipeline import prepare
from src.analysis import *
from src.visualization import *
def show(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format='png', bbox_inches='tight')
    display(Image(data=buffer.getvalue()))
    plt.close(figure)

## Build analytical tables

No raw source is overwritten. Output includes all raw fights with explicit scope flags.

In [2]:
data = prepare(save=False)
fights, appearances = cohort(data)
print(f"Cohort: {len(fights):,} fights, {fights.event_id.nunique():,} events, {appearances.fighter_id.nunique():,} fighters")
print(f"Date range: {fights.event_date.min():%Y-%m-%d} to {fights.event_date.max():%Y-%m-%d}")

Cohort: 8,820 fights, 783 events, 2,735 fighters
Date range: 1994-03-11 to 2026-08-08


## Data quality ledger

Zero-count checks remain visible in the exported ledger. Nonzero issues document actions rather than disappearing through row deletion.

In [3]:
display(data['quality'].query('affected_rows > 0'))
print('All fight rows preserved:', len(data['fights']) == len(load_raw()['fight']))
print('Unique fact IDs:', data['fights'].fight_id.is_unique)
print('Two appearances per fight:', len(data['appearances']) == 2*len(data['fights']))

,check,affected_rows,severity,action
1,Non-UFC event records,475,info,Retain in audit; exclude from headline UFC scope
2,Missing event locations,476,info,Unknown; exclude from geographic denominators
9,Historical profile weight above 400 lb,3,info,Preserve flagged historical open-weight outlie...
10,Repeated fighter names across IDs,16,info,Keep distinct IDs; use ID suffix in selectors
18,Repeated event and fighter pairing,2,info,"Keep IDs; possible tournament rematches, never..."
20,Implausible age in b corner,1,warning,Set to missing; preserve original DOB
85,Fights without complete round statistics,331,info,Retain outcomes; exclude incomplete totals fro...
86,Scrape errors with missing round stats,330,info,Coverage is nonrandom; display coverage by year
87,Master zero-filled fights without round records,330,info,Rebuild totals from round source; missing is n...


All fight rows preserved: True
Unique fact IDs: True
Two appearances per fight: True


## Scope and coverage

UFC main cards and TUF finales are distinguished from other promotions and feeder series. Coverage starts at 1994 for reporting; 1993 UFC bouts remain available to earlier-history features.

In [4]:
display(data['fights'].groupby(['is_ufc','in_scope']).agg(fights=('fight_id','size'), events=('event_id','nunique')))
coverage = data['fights'].groupby(['fight_year','is_ufc']).round_stats_complete.agg(['count','mean'])
display(coverage.head(20))

fights  events
is_ufc in_scope                
False  False       2613     475
True   False          8       1
       True        8820     783

count      mean
fight_year is_ufc                 
1993       True        8  1.000000
1994       True       31  0.935484
1995       True       40  0.850000
1996       True       43  0.837209
1997       False       8  1.000000
           True       41  0.951220
1998       False      21  1.000000
           True       25  0.840000
1999       False      29  1.000000
           True       44  1.000000
2000       False      57  1.000000
           True       43  1.000000
2001       False      65  0.876923
           True       40  1.000000
2002       False      87  0.804598
           True       53  1.000000
2003       False      77  1.000000
           True       41  1.000000
2004       False     117  0.991453
           True       39  1.000000

## Duration rules

Early formats include 10-minute rounds, overtime and no-time-limit bouts. A universal five-minute formula would be wrong.

In [5]:
from src.cleaning import fight_duration
examples = [('3 Rnd (10-5-5)', 2, '2:30'), ('1 Rnd + 2OT (15-3-3)', 3, '2:00'), ('No Time Limit', 1, '25:00')]
display(pd.DataFrame([dict(format=f, ending_round=r, clock=t, seconds=fight_duration(f,r,t)) for f,r,t in examples]))
display(fights.groupby('time_format').fight_duration_seconds.agg(['count','mean','max']))

,format,ending_round,clock,seconds
0,3 Rnd (10-5-5),2,2:30,750
1,1 Rnd + 2OT (15-3-3),3,2:00,1200
2,No Time Limit,1,25:00,1500


,count,mean,max
time_format,,,
1 Rnd (10),7,227.571429,600
1 Rnd (12),6,403.833333,720
1 Rnd (15),12,189.416667,900
1 Rnd (18),2,1080.000000,1080
1 Rnd (20),25,137.520000,747
1 Rnd (30),1,181.000000,181
1 Rnd + 2OT (15-3-3),20,495.500000,1260
1 Rnd + 2OT (24-3-3),1,1800.000000,1800
1 Rnd + OT (12-3),88,390.522727,900


## Inspect the feature timeline

Each current fight is excluded. Every fight on the same date shares the same earlier-date history because tournament order is unavailable.

In [6]:
sample_id = appearances.fighter_id.value_counts().index[0]
timeline = data['appearances'].query('fighter_id == @sample_id and is_ufc').sort_values('event_date')
display(timeline[['event_date','fighter_name','won','prior_ufc_fights','prior_wins','prior_win_streak','prior_sig_per_minute']].tail(12))
assert not appearances.duplicated(['fight_id','fighter_id']).any()
assert appearances.loc[~appearances.round_stats_complete,'sig_landed'].isna().all()

,event_date,fighter_name,won,prior_ufc_fights,prior_wins,prior_win_streak,prior_sig_per_minute
2022,2020-08-15,Jim Miller,False,35.0,21.0,1.0,2.812500
2023,2021-04-10,Jim Miller,False,36.0,21.0,0.0,2.732919
2018,2021-10-16,Jim Miller,True,37.0,21.0,0.0,2.651679
2016,2022-02-19,Jim Miller,True,38.0,22.0,1.0,2.700080
2017,2022-07-02,Jim Miller,True,39.0,23.0,2.0,2.776612
2012,2023-02-18,Jim Miller,False,40.0,24.0,3.0,2.791717
2015,2023-06-03,Jim Miller,True,41.0,24.0,0.0,2.854070
2013,2024-01-13,Jim Miller,True,42.0,25.0,1.0,2.858796
13455,2024-04-13,Jim Miller,False,43.0,26.0,2.0,2.958727
38,2024-11-16,Jim Miller,True,44.0,26.0,0.0,2.987956
